# Pretraining + MultiTaskResNetUNet smoke test

This is a **wiring test**, not a performance experiment. It uses two real local BRISC images resized to 64×64 and checks:

- the shared encoder produces correctly shaped classification and segmentation outputs;
- `total_loss = segmentation_loss + 0.25 × classification_loss` is finite;
- both heads and the shared encoder receive gradients and an optimizer step changes a weight;
- masked-denoising reconstruction can run one optimization step;
- pretrained U-Net weights transfer while the segmentation output head stays new;
- connected-component filtering and classifier gating work as inference-only postprocessing.

Passing this notebook means the code paths are connected correctly. It does **not** show that either model improves Dice.

In [1]:
from __future__ import annotations

import gc
import sys
import tempfile
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.optim import AdamW
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').exists(), 'Run this notebook from the repo or notebooks directory.'
assert (PROJECT_ROOT / 'brisc2025').exists(), 'Local BRISC data directory is missing.'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data import CLASS_NAMES
from models import MultiTaskResNetUNet, UNet
from pretrain_segmentation import corrupt_images, masked_reconstruction_loss
from segmentation_postprocessing import SegmentationPostprocessConfig, postprocess_masks
from train_segmentation import load_pretrained_unet, soft_dice_loss

SEED = 42
IMAGE_SIZE = 64
CLASSIFICATION_LOSS_WEIGHT = 0.25
DEVICE = torch.device('cpu')  # Small and predictable; no cloud or GPU required.
torch.manual_seed(SEED)

print(f'Project: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}; device: {DEVICE}')
print(f'Classification loss weight: {CLASSIFICATION_LOSS_WEIGHT}')

Project: /Users/louinsleung/Documents/brainTumor
PyTorch: 2.5.1; device: cpu
Classification loss weight: 0.25


## 1. Build a two-image real-data batch

The batch contains one tumor-positive image with its real mask and one no-tumor image with an empty mask. Albumentations is deliberately not required for this quick local check.

In [2]:
classification = pd.read_csv(PROJECT_ROOT / 'data' / 'classification_train.csv')
segmentation = pd.read_csv(PROJECT_ROOT / 'data' / 'segmentation_train.csv')
positive_row = segmentation.iloc[0]
normal_row = classification.loc[classification['tumor_label'] == 'no_tumor'].iloc[0]

def local_path(relative_path: str) -> Path:
    return PROJECT_ROOT / 'brisc2025' / Path(str(relative_path).replace('\\', '/'))

def load_image_tensor(relative_path: str, mode: str) -> torch.Tensor:
    with Image.open(local_path(relative_path)) as image:
        image = image.convert(mode)
        image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        return TF.to_tensor(image)

def load_mask_tensor(relative_path: str) -> torch.Tensor:
    with Image.open(local_path(relative_path)) as mask:
        mask = TF.resize(mask.convert('L'), [IMAGE_SIZE, IMAGE_SIZE], interpolation=InterpolationMode.NEAREST)
        return (TF.to_tensor(mask) > 0).float()

positive_rgb = load_image_tensor(positive_row['relative_path'], 'RGB')
normal_rgb = load_image_tensor(normal_row['relative_path'], 'RGB')
positive_gray = load_image_tensor(positive_row['relative_path'], 'L')
normal_gray = load_image_tensor(normal_row['relative_path'], 'L')
positive_mask = load_mask_tensor(positive_row['mask_relative_path'])
empty_mask = torch.zeros_like(positive_mask)

rgb_images = torch.stack([positive_rgb, normal_rgb]).to(DEVICE)
gray_images = torch.stack([positive_gray, normal_gray]).to(DEVICE)
masks = torch.stack([positive_mask, empty_mask]).to(DEVICE)
labels = torch.tensor([CLASS_NAMES.index(positive_row['tumor_label']), CLASS_NAMES.index('no_tumor')], device=DEVICE)

assert rgb_images.shape == (2, 3, IMAGE_SIZE, IMAGE_SIZE)
assert gray_images.shape == (2, 1, IMAGE_SIZE, IMAGE_SIZE)
assert masks.shape == (2, 1, IMAGE_SIZE, IMAGE_SIZE)
assert masks[0].any() and not masks[1].any()
pd.DataFrame([
    {'sample': 'tumor-positive', 'label': positive_row['tumor_label'], 'mask_pixels': int(masks[0].sum())},
    {'sample': 'no-tumor', 'label': 'no_tumor', 'mask_pixels': int(masks[1].sum())},
])

,sample,label,mask_pixels
0,tumor-positive,glioma,263
1,no-tumor,no_tumor,0


## 2. MultiTaskResNetUNet forward, losses, gradients, and update

The segmentation term uses symmetric BCE + soft Dice for this comparison smoke test. The previous false-positive-aware training loss is intentionally not mixed into this experiment.

In [3]:
multitask_model = MultiTaskResNetUNet(pretrained=False).to(DEVICE)
multitask_model.train()
multitask_optimizer = AdamW(multitask_model.parameters(), lr=1e-4, weight_decay=1e-4)
encoder_weight_before = multitask_model.encoder.stem[0].weight.detach().clone()

multitask_optimizer.zero_grad(set_to_none=True)
class_logits, mask_logits = multitask_model(rgb_images)
assert class_logits.shape == (2, len(CLASS_NAMES))
assert mask_logits.shape == masks.shape

segmentation_bce = F.binary_cross_entropy_with_logits(mask_logits, masks)
segmentation_dice = soft_dice_loss(mask_logits, masks)
segmentation_loss = 0.5 * segmentation_bce + 0.5 * segmentation_dice
classification_loss = F.cross_entropy(class_logits, labels)
total_loss = segmentation_loss + CLASSIFICATION_LOSS_WEIGHT * classification_loss
assert torch.isfinite(total_loss)
total_loss.backward()

def gradient_l1(parameters) -> float:
    return float(sum(p.grad.detach().abs().sum() for p in parameters if p.grad is not None))

gradient_summary = {
    'shared_encoder': gradient_l1(multitask_model.encoder.parameters()),
    'classification_head': gradient_l1(multitask_model.classifier.parameters()),
    'segmentation_head': gradient_l1(multitask_model.segmentation_head.parameters()),
}
assert all(value > 0 for value in gradient_summary.values())
multitask_optimizer.step()
assert not torch.equal(encoder_weight_before, multitask_model.encoder.stem[0].weight.detach())

multitask_result = {
    'segmentation_loss': float(segmentation_loss.detach()),
    'classification_loss': float(classification_loss.detach()),
    'weighted_classification_term': float((CLASSIFICATION_LOSS_WEIGHT * classification_loss).detach()),
    'total_loss': float(total_loss.detach()),
    **{f'{name}_gradient_l1': value for name, value in gradient_summary.items()},
}
pd.Series(multitask_result, name='value')

segmentation_loss                      0.796375
classification_loss                    1.743626
weighted_classification_term           0.435907
total_loss                             1.232282
shared_encoder_gradient_l1         20026.410156
classification_head_gradient_l1      143.715714
segmentation_head_gradient_l1          2.696271
Name: value, dtype: float64

### Tiny fixed-batch overfit check

A smoke test should also establish that both task losses can move downward. We deliberately reuse the same two images for a few steps. Final loss only needs to be lower than initial loss; individual steps need not be monotonic. This checks optimization wiring, not generalization.

In [4]:
loss_trace = []
for step in range(8):
    multitask_optimizer.zero_grad(set_to_none=True)
    step_class_logits, step_mask_logits = multitask_model(rgb_images)
    step_segmentation_loss = (
        0.5 * F.binary_cross_entropy_with_logits(step_mask_logits, masks)
        + 0.5 * soft_dice_loss(step_mask_logits, masks)
    )
    step_classification_loss = F.cross_entropy(step_class_logits, labels)
    step_total_loss = (
        step_segmentation_loss
        + CLASSIFICATION_LOSS_WEIGHT * step_classification_loss
    )
    step_total_loss.backward()
    multitask_optimizer.step()
    loss_trace.append({
        'step': step + 1,
        'segmentation_loss': float(step_segmentation_loss.detach()),
        'classification_loss': float(step_classification_loss.detach()),
        'total_loss': float(step_total_loss.detach()),
    })

loss_trace = pd.DataFrame(loss_trace)
assert loss_trace.iloc[-1]['segmentation_loss'] < loss_trace.iloc[0]['segmentation_loss']
assert loss_trace.iloc[-1]['classification_loss'] < loss_trace.iloc[0]['classification_loss']
assert loss_trace.iloc[-1]['total_loss'] < loss_trace.iloc[0]['total_loss']
loss_trace

,step,segmentation_loss,classification_loss,total_loss
0,1,0.778693,0.283361,0.849534
1,2,0.765602,0.074796,0.784301
2,3,0.752392,0.036360,0.761483
3,4,0.742739,0.022193,0.748287
4,5,0.734940,0.015182,0.738735
5,6,0.728521,0.011258,0.731335
6,7,0.722514,0.008674,0.724682
7,8,0.716844,0.006900,0.718569


In [5]:
# Release the larger multitask graph before the U-Net pretraining check.
del multitask_optimizer, multitask_model, class_logits, mask_logits, total_loss
del step_class_logits, step_mask_logits, step_total_loss
gc.collect()

0

## 3. Masked-denoising pretraining and checkpoint transfer

One reconstruction step verifies corruption, masked-only loss, backpropagation, and transfer into a new segmentation U-Net. The reconstruction output head must not be transferred.

In [6]:
pretraining_model = UNet(n_channels=1, n_classes=1, bilinear=False).to(DEVICE)
pretraining_model.train()
pretraining_optimizer = AdamW(pretraining_model.parameters(), lr=1e-4, weight_decay=1e-4)
generator = torch.Generator().manual_seed(SEED)
corrupted, patch_mask = corrupt_images(
    gray_images.cpu(), patch_size=8, mask_ratio=0.35, noise_std=0.05, generator=generator
)
corrupted = corrupted.to(DEVICE)
patch_mask = patch_mask.to(DEVICE)

pretraining_optimizer.zero_grad(set_to_none=True)
reconstruction = pretraining_model(corrupted)
reconstruction_loss = masked_reconstruction_loss(reconstruction, gray_images, patch_mask)
assert reconstruction.shape == gray_images.shape
assert torch.isfinite(reconstruction_loss)
reconstruction_loss.backward()
pretraining_gradient = gradient_l1(pretraining_model.parameters())
assert pretraining_gradient > 0
pretraining_optimizer.step()

segmentation_model = UNet(n_channels=1, n_classes=1, bilinear=False).to(DEVICE)
new_output_head = segmentation_model.outc.conv.weight.detach().clone()
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = Path(directory) / 'masked_pretraining_smoke.pt'
    torch.save({
        'model_state_dict': pretraining_model.state_dict(),
        'epoch': 1,
        'val_masked_l1': float(reconstruction_loss.detach()),
        'config': {'bilinear': False},
        'pretraining_source_split': 'train',
    }, checkpoint_path)
    transfer_metadata = load_pretrained_unet(segmentation_model, checkpoint_path, DEVICE)

assert torch.equal(
    segmentation_model.inc.double_conv[0].weight,
    pretraining_model.inc.double_conv[0].weight,
)
assert torch.equal(segmentation_model.outc.conv.weight, new_output_head)
assert transfer_metadata['output_head_reinitialized']

pretraining_result = {
    'masked_fraction': float(patch_mask.float().mean()),
    'masked_reconstruction_loss': float(reconstruction_loss.detach()),
    'pretraining_gradient_l1': pretraining_gradient,
    'transferred_tensor_count': transfer_metadata['transferred_tensor_count'],
    'output_head_reinitialized': transfer_metadata['output_head_reinitialized'],
}
pd.Series(pretraining_result, name='value')

masked_fraction                   0.34375
masked_reconstruction_loss       0.357757
pretraining_gradient_l1       3209.925049
transferred_tensor_count              116
output_head_reinitialized            True
Name: value, dtype: object

In [7]:
del pretraining_optimizer, pretraining_model, segmentation_model, reconstruction
gc.collect()

8

## 4. Inference-only postprocessing

This constructed example confirms that a tiny isolated component is removed, a larger region survives, and classifier gating can clear a complete mask. No model training occurs here.

In [8]:
example_masks = torch.zeros(2, 1, 8, 8, dtype=torch.bool)
example_masks[0, 0, 0, 0] = True       # one-pixel false-positive component
example_masks[0, 0, 3:6, 3:6] = True   # nine-pixel region that should survive
example_masks[1, 0, 2:6, 2:6] = True   # cleared by the classifier gate
config = SegmentationPostprocessConfig(
    min_component_pixels=4,
    no_tumor_gate_threshold=0.90,
)
processed_masks, metadata = postprocess_masks(
    example_masks, config, no_tumor_probabilities=[0.10, 0.95]
)
assert int(processed_masks[0].sum()) == 9
assert not processed_masks[1].any()
assert metadata[0]['removed_component_count'] == 1
assert metadata[1]['classifier_gated']
postprocessing_result = {
    'sample_0_pixels_before': int(example_masks[0].sum()),
    'sample_0_pixels_after_filter': int(processed_masks[0].sum()),
    'sample_1_pixels_before': int(example_masks[1].sum()),
    'sample_1_pixels_after_gate': int(processed_masks[1].sum()),
}
pd.Series(postprocessing_result, name='value')

sample_0_pixels_before          10
sample_0_pixels_after_filter     9
sample_1_pixels_before          16
sample_1_pixels_after_gate       0
Name: value, dtype: int64

## 5. Final smoke-test result

In [9]:
checks = pd.DataFrame([
    {'check': 'Real local image + mask loading', 'status': 'PASS'},
    {'check': 'MultiTaskResNetUNet forward shapes', 'status': 'PASS'},
    {'check': 'Weighted multitask loss is finite', 'status': 'PASS'},
    {'check': 'Both heads and shared encoder receive gradients', 'status': 'PASS'},
    {'check': 'Optimizer changes shared encoder weights', 'status': 'PASS'},
    {'check': 'Masked-denoising optimization step', 'status': 'PASS'},
    {'check': 'Pretraining checkpoint transfer', 'status': 'PASS'},
    {'check': 'Reconstruction output head stays excluded', 'status': 'PASS'},
    {'check': 'Filtering and gating postprocessing', 'status': 'PASS'},
])
assert (checks['status'] == 'PASS').all()
print('SMOKE TEST PASSED — code paths are ready for a small training run.')
checks

SMOKE TEST PASSED — code paths are ready for a small training run.


,check,status
0,Real local image + mask loading,PASS
1,MultiTaskResNetUNet forward shapes,PASS
2,Weighted multitask loss is finite,PASS
3,Both heads and shared encoder receive gradients,PASS
4,Optimizer changes shared encoder weights,PASS
5,Masked-denoising optimization step,PASS
6,Pretraining checkpoint transfer,PASS
7,Reconstruction output head stays excluded,PASS
8,Filtering and gating postprocessing,PASS
